<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/transformers_agents_multiagents/Langchain_Learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#pip install -qU langchain  langchain-cohere langchain_community

In [ ]:
import getpass
import os
from langchain_cohere import ChatCohere
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

C:\Users\Ihechi Festus\Documents\ML\ml_env\Lib\site-packages\pydantic\_internal\_config.py:341: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'smart_union' has been removed
  warnings.warn(message, UserWarning)


In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "test-project"
os.environ["USER_AGENT"] = "LangChain/1.0.0"

### Provide API Keys.
P.S: You can add them to environment variables instead of typing them out.

In [ ]:
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()

 ········


In [ ]:
os.environ["COHERE_API_KEY"] = getpass.getpass()

 ········


In [ ]:
os.environ["TAVILY_API_KEY"] = getpass.getpass()

 ········


### Define the Model

In [ ]:
model = ChatCohere(model="command-r-plus")

In [ ]:
store = {}

In [ ]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    model, get_session_history,
)

In [ ]:
# Build context Prompt
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    model, retriever, contextualize_q_prompt
)

NameError: name 'retriever' is not defined

### Agents
Agents leverage the reasoning capabilities of LLMs to make decisions during execution. Using agents allow you to offload some discretion over the retrieval process. They offer advantages in this context:
- Agents generate the input to the retriever directly, without necessarily needing us to explicitly build in contextualization.
- Agents can execute multiple retrieval steps in service of a query, or refrain from executing a retrieval step altogether(eg, in response to a generic greeting from a user)

Retrieval tool
Agents can access "tools" and manage their execution. In this case, we will convert our retriever into a LangChain tool to be wielded by the agent:

In [ ]:
from langchain.tools.retriever import create_retriever_tool

tool = create_retriever_tool(
    retriever,
    "blog_post_retriever",
    "Searches and returns excerpts from the Autonomous Agents blog post.",
)
tools = [tool]

In [ ]:
# Build the agent
memory = MemorySaver()
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_react_agent(model, tools, checkpointer=memory)

In [ ]:
query = "What is Task Decomposition?"

for event in agent_executor.stream(
    {"messages": [HumanMessage(content=query)]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

Note that if we input a query that does not require a retrieval step(Such as a simple greeting), the agent does not execute one.
Above, instead of inserting our query verbatim into the tool, the agent stripped unnecessary words like "what" and "is".
This same principle allows the agent to use the context of the conversation when necessary:

In [ ]:
query = "What according to the blog post are common ways of doing it? redo the search"

for event in agent_executor.stream(
    {"messages": [HumanMessage(content=query)]},
    config=config,
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

In [ ]:
# Use the agent
config = {"configurable": {"thread_id": "abc123"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi im bob! and i live in sf")]}, config
):
    print(chunk)
    print("----")


In [ ]:
from langchain_core.runnables import ConfigurableFieldSpec


def get_session_history(user_id: str, conversation_id: str):
    return SQLChatMessageHistory(f"{user_id}--{conversation_id}", "sqlite:///memory.db")


with_message_history = RunnableWithMessageHistory(
    runnable,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",
            annotation=str,
            name="User ID",
            description="Unique identifier for the user.",
            default="",
            is_shared=True,
        ),
        ConfigurableFieldSpec(
            id="conversation_id",
            annotation=str,
            name="Conversation ID",
            description="Unique identifier for the conversation.",
            default="",
            is_shared=True,
        ),
    ],
)

with_message_history.invoke(
    {"language": "italian", "input": "hi im bob!"},
    config={"configurable": {"user_id": "123", "conversation_id": "1"}},
)

In [ ]:
chunks = []
async for chunk in with_message_history.astream("what color is the sky?"):
    chunks.append(chunk)
    print(chunk.content, end="|", flush=True)

In [ ]:
chunks[0] + chunks[1] + chunks[2] + chunks[3] + chunks[4]

In [ ]:
import time

In [ ]:
# Import relevant functionality
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

# Create the agent
memory = MemorySaver()
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_react_agent(model, tools, checkpointer=memory)

# Use the agent
config = {"configurable": {"thread_id": "abc123"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi im bob! and i live in sf")]}, config
):
    print(chunk)
    print("----")


In [ ]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    model, retriever, contextualize_q_prompt
)

In [ ]:
from langchain import hub

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

In [ ]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

default_system_prompt = (
    "You are a helpful assistant"
)

default_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", default_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

summarization_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="chat_history"),
        (
            "user",
            "Distill the above chat messages into a single summary message. Include as many specific details as you can.",
        ),
    ]
)

In [ ]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
from collections import defaultdict

In [ ]:
store

In [ ]:
get_session_history("b", "c")

In [ ]:
store

In [ ]:
store = defaultdict(dict, {})

In [ ]:
c = store["a"]["b"]

In [ ]:
c.add_message("I have a ph.d in Jolly")

In [ ]:
c

In [ ]:
summarize_messages({"input": "hello"}, {"configurable": {"user_id": "a",
                    "conversation_id": "b"
                                                        }
                                       }
                  )

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

# Create the agent
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_react_agent(model, tools)

In [ ]:
from langchain_core.runnables import ConfigurableFieldSpec
store = defaultdict(dict, {})

def get_session_history(user_id: str, conversation_id: str) -> BaseChatMessageHistory:
    if conversation_id not in store[user_id]:
        store[user_id][conversation_id] = ChatMessageHistory()

    return store[user_id][conversation_id]

def summarize_messages(chain_input, config):
    user_id = config.get("configurable", {}).get("user_id")
    conversation_id = config.get("configurable", {}).get("conversation_id")
    checkpoint = 10

    session_history = get_session_history(user_id, conversation_id)
    stored_messages = session_history.messages

    # summarize conversations if len of conversation greater than checkpoint.
    # offset by 1 to account for AImessage which holds previous summaries
    if len(stored_messages) >= checkpoint + 1:
        summarization_chain = summarization_prompt | model

        summary_message = summarization_chain.invoke(
            {
                "chat_history": stored_messages
            }
        )
        session_history.clear()
        session_history.add_message(summary_message)

    return session_history.messages

agent_with_chat_history  = RunnableWithMessageHistory(
    default_prompt | model,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",
            annotation=str,
            name="User ID",
            description="Unique identifier for the user.",
            default="",
            is_shared=True,
        ),
        ConfigurableFieldSpec(
            id="conversation_id",
            annotation=str,
            name="Conversation ID",
            description="Unique identifier for the conversation.",
            default="",
            is_shared=True,
        ),
    ],
)

chain_with_summarization = (
    RunnablePassthrough.assign(chat_history=lambda chain_input, config: summarize_messages(chain_input, config))
    | RunnablePassthrough.assign(m=see_input)
    | agent_with_chat_history
)

# agent_with_chat_history.invoke(
#     {"language": "italian", "input": "hi im bob!"},
#     config={"configurable": {"user_id": "123", "conversation_id": "1"}},
# )

In [ ]:
def see_input(chain_input):
    print("Chain input: ", chain_input)
    return chain_input

In [ ]:
chain_with_summarization.invoke(
    {"input": "Alright, that is understandable", },
    config={"configurable": {"user_id": "a", "conversation_id": "b"}}
)

In [ ]:
c = store["a"]["b"]

In [ ]:
for message in c.messages:
    if isinstance(message, AIMessage):
        prefix = "AI"
    else:
        prefix = "User"

    print(f"{prefix}: {message.content}\n")

#### Statuful Management of Chat History
LangGraph implements a built-in persistence layer, making it ideal for chat applications that support multiple conversational turns.
Wrapping our chat model in a minimal LangGraph application allows us to automatically persist the message history, simplifying the development of multi-turn applications.

LangGraph comes with a simple in-memory checkpoint.

[Detailed documentation of the persistence layer, including how to use different persistence backends(eg. SQLite, PostGres)](https://langchain-ai.github.io/langgraph/concepts/persistence/)


In [ ]:
from typing import Sequence

from langchain_core.messages import BaseMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict

# We define a dict representing the state of the application.
# This state has the same input and output keys as `rag_chain`.
class State(TypedDict):
    input: str
    chat_history: Annotated[Sequence[BaseMessage], add_messages]
    context: str
    answer: str

# We then define a simple node that runs the `rag_chain`
# The `return` values of the node update the graph state, so here we just
# update the chat history with the input message and response.
def call_model(state: State):
    response = rag_chain.invoke(state)
    return {
        "chat_history": [
            HumanMessage(state["input"]),
            AIMessage(response["answer"]),
        ],
        "context": response["context"],
        "answer": response["answer"],
    }

# Our graph consists only of one node:
workflow = StateGraph(state_schema=State)
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

# Finally, we compile the graph with a checkpointer object.
# This persists the state, in this case in memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)



This application out-of-the-box supports multiple conversation threads. We pass in a configuration dict specifying a unique identifier for a thread to control what thread is run. This enables the application to support interactions with multiple users.

In [ ]:
result = app.invoke(
    {"input": "What is Task Decomposition?"},
    config=config,
    )

print(result["answer"])

result = app.invoke(
    {"input": "What is one way of doing it?"},
    config=config,
)
print(result["answer"])


# The conversation history can be inspected via the state of the application:
chat_history = app.get_state(config).values["chat_history"]
for message in chat_history:
    message.pretty_print()
